# Mining data - cleaning and demonstration

This sheet here takes the public list of lithium mines and fills the missing values with some synthetic values. It also creates some simple visualisations, and gives a demonstration of the mining cost curves which can be inferred from them.

This is the process by which **existing** mines deposits (operating, under construction, and discovered) are initiated. This does not cover the process of further exploration leading to new discoveries.

The dataset is incomplete. This is likely to be most significant for early stage mines. 

## Data used

The initial dataset used to initialise the mines is from the below source:

https://github.com/johannahansel/lithium-datasets

Eventually, I would like to pair this dataset with the following dataset:

https://ecl.earthchem.org/view.php?id=3329
https://ui.adsabs.harvard.edu/abs/2024ieda.data..140B/abstract


This provides a snapshot of all the different deposits around the world. I could match on the longitude and latitude to add more operational detail. This would allow me to add more detail ore grade and project type information.

## Assumptions

1. Mining capacity limits annual extraction or ore or raw brine. On-site concentration has sufficient capacity to process this material within the model period. Stockpiling and processing delays are initially omitted.

In [ ]:

import pandas as pd
import sys, pathlib
import plotly.graph_objects as go
import numpy as np


seed = 1
rng = np.random.default_rng(seed)

sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # adds 02_Model/

# UNITS

LCE_PER_LI = 5.323
LCE_PER_LI2O = 2.473
BRINE_DENSITY = 1.2 # t per m3

# ASSUMPTIONS - for now this is hardcoded, but will eventually come from the config file or a database.

PRE_CONSTRUCTION = {"minimum": 3, "maximum": 10, "median": 5}
CONSTRUCTION = {"minimum": 1, "maximum": 5, "median":3}
LIFESPAN = {"minimum":10, "maximum": 50, "median": 20}

# MODEL

BASE_YEAR = 2024
START_YEAR = 2026
ROLL_FORWARD_UTIL = 1.0 # utilisation rate assumed for periods before start date


# Grades

GRADE = {
    "hard_rock" : {"minimum" : 0.6, "maximum" : 2, "median": 1.2, "units" : "% Li2O"},
    "brine": {"minimum" : 200, "maximum" : 2000, "median": 600, "units" : "mg/L Li"}
}

GRADE_DRIFT = {"operation" : 1.15, "construction" : 1.08, "pre-construction" : 0.92}

MINING_RECOVERY = {
    "hard_rock": 0.65,
    "brine" : 0.55
}

COST_PER_T_MATERIAL = {
    "hard_rock" : 125.0,
    "brine" : 5.0,
    "units" : "USD/t"}

COST_SPREAD = 0.15 # sigma of the lognormal space for the cost. Choose a lower value here so that there is in minimal variation in the spread.


In [351]:
# import the data
data_path = pathlib.Path.cwd().parent / "data" / "master_database_projects_UPDATED.csv"
df = pd.read_csv(data_path)

export_path = pathlib.Path.cwd().parent / "data" / f"synthetic_mine_seed{seed}.csv"

# Clean the data

- Drop columns I don't want just get.
- Clean the coordinate

In [ ]:

df.columns = df.columns.str.strip()  # remove whitespace from column names

df = df[["Property Name", "Country/ Region",  "Development Stage", "Primary Reserves and Resources", 
         "Unit", "Project Type", "Coordinates", "Construction Start", "Production Start", "Closure", 
         "Reserves & Resources As Of Date" ]]

df = df.rename(columns={
    "Property Name": "name", "Country/ Region": "country", 
    "Primary Reserves and Resources": "reserves", "Unit": "unit","Project Type": "project_type", "Coordinates": "coordinates", "Production Start": "production_start", 
    "Construction Start": "construction_start", "Closure": "closure", "Reserves & Resources As Of Date" : "reserves_as_of"})

In [353]:
# Add consistent stages

stage = df["Development Stage"].str.strip()

pre_construction = {
    "Target Outline", "Exploration", "Advanced Exploration", "Grassroots",
    "Reserves Development", "Reserves Development - Prefeas/Scoping",
    "Prefeas/Scoping", "Feasibility", "Feasibility Started", "Feasibility Complete",
}
construction = {
    "Construction Planned", "Construction Started", "Under Construction",
    "Commissioning", "Preproduction",
    "Preproduction - Construction Planned", "Preproduction - Construction Started",
    "Preproduction - Commissioning", "Preproduction - Expansion",  # judgment call: treated as still building out
}
operation = {
    "Operating", "Production - Operating", "Expansion", "Limited Production",
    "Satellite",  # judgment call: satellite deposit of an active mine
    "Preproduction - Operating",  # judgment call: contradictory label in source data, treated as operating
}

stage_group = pd.Series("pre-construction", index=df.index)
stage_group[stage.isin(construction)] = "construction"
stage_group[stage.isin(operation)] = "operation"
df["stage_group"] = stage_group

### Just keep the complete data for now

At this stage I will drop all the observations for where we don't have the complete data, namely the ones without primary reserves and resources for the non operational / under construction mines.


In [354]:
coords = df["coordinates"].str.split(",", expand=True)
df["lat"] = pd.to_numeric(coords[0], errors="coerce")
df["lon"] = pd.to_numeric(coords[1], errors="coerce")

df = df.drop(columns=["Development Stage", "coordinates"])

df["reserves"] = pd.to_numeric(df["reserves"], errors="coerce")

In [355]:
# drop the rows where stage_group is pre-construction and reserves is NaN
DROP_INCOMPLETE = False

if DROP_INCOMPLETE:
    drop_mask = (df["stage_group"] == "pre-construction") & df["reserves"].isna()
    df = df[~drop_mask]


## Generate some useful values

For each of the mines with missing values I need to create some data to fill in the missing data, using some assumptions

In [356]:
df["construction_start"] = pd.to_numeric(df["construction_start"], errors="coerce")
df["production_start"] = pd.to_numeric(df["production_start"], errors="coerce")
df["closure"] = pd.to_numeric(df["closure"], errors="coerce")
df["reserves_as_of_year"] = pd.to_datetime(df["reserves_as_of"], errors="coerce").dt.year


constr_dur = (df["production_start"] - df["construction_start"]).dropna()
lifespan = (df["closure"] - df["production_start"]).dropna()

/var/folders/nr/0xxnbd0s1w742fz536j6wtrwq03qpn/T/ipykernel_86324/1871021373.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["reserves_as_of_year"] = pd.to_datetime(df["reserves_as_of"], errors="coerce").dt.year


In [ ]:


def lce_fraction(grade_value, deposit_type):
    """t LCE per t of material moved."""
    if deposit_type == "hard_rock":
        return (grade_value / 100) * LCE_PER_LI2O * MINING_RECOVERY["hard_rock"]
    t_li_per_m3 = grade_value * 1e-6
    return (t_li_per_m3 * LCE_PER_LI / BRINE_DENSITY) * MINING_RECOVERY["brine"]

# lognormal fit for missing data

def fit_lognormal(x):
    x = x.dropna()
    x = x[x>0]
    log_x = np.log(x)
    return log_x.mean(), log_x.std()

reserves_mu, reserves_sigma = fit_lognormal(df["reserves"])
constr_mu, constr_sigma = fit_lognormal(constr_dur)
life_mu, life_sigma = fit_lognormal(lifespan)

def fill_with_draws(series, mu, sigma, rng):
    missing = series.isna()
    draws = rng.lognormal(mu, sigma, size=missing.sum())
    return series.mask(missing, pd.Series(draws, index=series.index[missing]))



# Filling the rest of the data

Future updates:

    - change the pre-construction years to be impacted by the risk score
    - have the probability of completion also impacted by the risk score

In [358]:
def sample_triangular(assumptions, size, rng):
    return rng.triangular(assumptions["minimum"], assumptions["median"], assumptions["maximum"], size=size)

# no real data exists for pre-construction duration, so every row is synthetic
df["pre_construction_years"] = sample_triangular(PRE_CONSTRUCTION, len(df), rng)

# construction: use the real duration where both dates exist, else draw from assumptions
real_construction = df["production_start"] - df["construction_start"]
synth_construction = pd.Series(sample_triangular(CONSTRUCTION, len(df), rng), index=df.index)
df["construction_years"] = real_construction.where(real_construction.notna(), synth_construction)

# lifespan

real_lifespan = df["closure"] - df["production_start"]
synth_lifespan = pd.Series(sample_triangular(LIFESPAN, len(df), rng), index=df.index)
df["lifespan_years"] = real_lifespan.where(real_lifespan.notna(), synth_lifespan)
df["reserves_filled"] = fill_with_draws(df["reserves"], reserves_mu, reserves_sigma, rng)



remaining_years = (df["closure"] - df["reserves_as_of_year"]).where(df["stage_group"] =="operation")
denominator = df["lifespan_years"].where(remaining_years.isna(), remaining_years)


# capacity 
df["capacity"] = df["reserves_filled"] / denominator



# capacity

# df["capacity"] = df["reserves_filled"] / df["lifespan_years"]

## Adding the types to the projects that are missing it

A number of the mines are missing the project type data. For those ones I impute it based just on the location.

I simplify it to just brine and hard rock for now as well.

Later, I will want to differentiate the costs, grade etc. on the types in a bit more depth. This is a good first simplification though.

In [359]:
DEPOSIT_TYPES = ["brine", "hard_rock"]

def bucket_project_type(t):
    if pd.isna(t):
        return None
    t = t.strip().lower()
    if any(k in t for k in ("brine", "dle", "geothermal", "in-situ leach")):
        return "brine"
    if any(k in t for k in ("open pit", "underground", "placer", "tailings", "stock pile")):
        return "hard_rock"
    return None

df["deposit_type_known"] = df["project_type"].map(bucket_project_type)

known = df.dropna(subset=["deposit_type_known"])
country_mix = (known.groupby("country")["deposit_type_known"].value_counts(normalize=True)
               .unstack(fill_value=0.0).reindex(columns=DEPOSIT_TYPES, fill_value=0.0))
global_mix = known["deposit_type_known"].value_counts(normalize=True).reindex(DEPOSIT_TYPES, fill_value=0.0)

def impute_type(country, rng):
    p = country_mix.loc[country] if country in country_mix.index else global_mix
    if p.sum() == 0:
        p = global_mix
    return rng.choice(DEPOSIT_TYPES, p=(p / p.sum()).values)

miss = df["deposit_type_known"].isna()
df["deposit_type"] = df["deposit_type_known"]
df.loc[miss, "deposit_type"] = [impute_type(c, rng) for c in df.loc[miss, "country"]]
df["deposit_type_imputed"] = miss

## Add the grade distribution in

Next I will draw the grade distributions and calculate the derived columsn (e.g. cost per t LCE.)

I add a slight stage drift as well - it is likely that the already operational mines have higher grades (or lower costs) than the others. Equally, the under construction probably have higher grades than the pre-construction.

This is a large simplification - there are a myriad of other factors, such as CAPEX costs, institutional / regulatory environment, and some ore dynamics (e.g. sometimes lower ore grade may have lower processing costs because of a lack of certain coproducts). I will revisit these factos at a latter point so as to add more realism.

In [360]:
df["grade_base"] = np.nan
for dt in DEPOSIT_TYPES:
    m = df["deposit_type"] == dt
    df.loc[m, "grade_base"] = sample_triangular(GRADE[dt], int(m.sum()), rng)

df["grade"] = df["grade_base"] * df["stage_group"].map(GRADE_DRIFT)
df["grade_unit"] = df["deposit_type"].map({"hard_rock": "pct_Li2O", "brine": "mg_per_L_Li"})


## Add the recovery efficiency in

For now, the recovery efficiency rate is the same for all projects by type. Later, this may change, so we add it to the table.

This will need refinement to add some randomness AND to account for different pathways, regions etc.

In [ ]:
df["recovery_efficiency"] = df["deposit_type"].map(MINING_RECOVERY)

## Calculate the costs per LCE etc.

In [362]:
df["lce_fraction"] = np.where(df["deposit_type"] == "hard_rock",
                              (df["grade"] / 100) * LCE_PER_LI2O * df["recovery_efficiency"],
                              (df["grade"] / 100 * 1e-6 * LCE_PER_LI / BRINE_DENSITY) * df["recovery_efficiency"])

# Calculating the costs

In mining economics, the C1 cost is often used for production decisions. C1 includes operating expenditure, including osme overheads, so it is not strictly the cost of producing one additional tonner. For example, Rio Tinto's definition includes site admininistration alongside mining and processing costs.

For the first version however I assume that at the assumed utilisation rate, unit operating cash cost approximates marginal produciton cost. 

This supports a simple production rule: produce when the revenue per unit of the mine's output covers its unit operating cost. 

## Simple plot with the exact location

In [363]:
fallback_size = df.loc[df["stage_group"] != "pre-construction", "reserves_filled"].quantile(0.25)
df["reserves_plot"] = df["reserves_filled"].fillna(fallback_size)
df["reserves_opacity"] = df["reserves_filled"].notna().map({True: 0.9, False: 0.35})

colors = {"pre-construction": "#4C78A8", "construction": "#F58518", "operation": "#54A24B"}

fig = go.Figure()
for stage_name, color in colors.items():
    sub = df[df["stage_group"] == stage_name]
    fig.add_trace(go.Scattermapbox(
        lat=sub["lat"], lon=sub["lon"],
        mode="markers",
        marker=dict(size=sub["reserves_plot"] / df["reserves_plot"].max() * 30,
            color=color, opacity=sub["reserves_opacity"]),
        name=stage_name,
        text=sub["name"],
    ))
fig.update_layout(mapbox_style="open-street-map", mapbox_zoom=1)
fig.show()


/var/folders/nr/0xxnbd0s1w742fz536j6wtrwq03qpn/T/ipykernel_86324/165718491.py:10: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/nr/0xxnbd0s1w742fz536j6wtrwq03qpn/T/ipykernel_86324/165718491.py:10: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(
/var/folders/nr/0xxnbd0s1w742fz536j6wtrwq03qpn/T/ipykernel_86324/165718491.py:10: DeprecationWarning: *scattermapbox* is deprecated! Use *scattermap* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig.add_trace(go.Scattermapbox(


## Same plot again with capacity instead of reserves

### Export the file to csv



In [364]:
df.to_csv(export_path, index=False)
